# 3. ChemBERTa model Comparison

## pre-trained model vs baseline model

In [1]:
!pip install rdkit transformers torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 11.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026
!pwd

Cloning into 'laidd-2026'...
remote: Enumerating objects: 34, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 34 (delta 5), reused 20 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (34/34), 17.95 KiB | 8.98 MiB/s, done.
Resolving deltas: 100% (5/5), done.
/content/laidd-2026
/content/laidd-2026


In [3]:
from src.tools.data_prep import load_tox21_clean

data = load_tox21_clean()
print("불러오기 성공, task 개수:", len(data['task_cols']))

[14:44:30] WARNING: not removing hydrogen atom without neighbors
[14:44:30] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:44:31] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:44:31] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:44:31] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:44:31] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:44:31] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:44:31] Explicit valence for atom # 16 Al, 6, is greater than permitted
[14:44:32] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[14:44:32] WARNING: not removing hydrogen atom without neighbors


불러오기 성공, task 개수: 12


In [4]:
import torch
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()  # 학습 모드가 아니라 추론 모드로 고정

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("사용 디바이스:", device)

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/9.43k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.21k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.decoder.bias      | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  179MB            

model.safetensors: downloading bytes:           |  0.00B            

사용 디바이스: cuda


In [5]:
import numpy as np
from tqdm import tqdm

def get_chemberta_embeddings(smiles_list, batch_size=64):
    embeddings = []
    for i in tqdm(range(0, len(smiles_list), batch_size)):
        batch = smiles_list[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            # [CLS] 토큰(문장 전체를 요약하는 첫 토큰)의 벡터를 분자 표현으로 사용
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)

    return np.concatenate(embeddings, axis=0)

In [6]:
%%writefile src/tools/data_prep.py
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.model_selection import train_test_split

TOX21_URL = "https://deepchemdata.s3-us-west-1.amazonaws.com/datasets/tox21.csv.gz"

_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

def _is_valid_smiles(smiles):
    return Chem.MolFromSmiles(smiles) is not None

def _smiles_to_ecfp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    fp = _generator.GetFingerprintAsNumPy(mol)
    return np.array(fp)

def load_tox21_clean(test_size=0.3, valid_ratio=0.5, random_state=42):
    df = pd.read_csv(TOX21_URL)
    df['valid'] = df['smiles'].apply(_is_valid_smiles)

    n_total, n_valid = len(df), df['valid'].sum()
    print(f"전체: {n_total}개, 파싱 성공: {n_valid}개, 파싱 실패(제외): {n_total - n_valid}개")

    invalid_smiles = df[~df['valid']]['smiles'].tolist()
    df_clean = df[df['valid']].reset_index(drop=True)

    task_cols = [c for c in df.columns if c not in ['smiles', 'mol_id', 'valid']]

    y = df_clean[task_cols].fillna(0).values.astype(np.float32)
    w = (~df_clean[task_cols].isna()).values.astype(np.float32)
    X = np.stack(df_clean['smiles'].apply(_smiles_to_ecfp).values)
    smiles_arr = df_clean['smiles'].values  # SMILES 원본, 순서는 X/y/w와 동일하게 정렬됨

    indices = np.arange(len(X))
    train_idx, temp_idx = train_test_split(indices, test_size=test_size, random_state=random_state)
    valid_idx, test_idx = train_test_split(temp_idx, test_size=valid_ratio, random_state=random_state)

    return {
        'X_train': X[train_idx], 'y_train': y[train_idx], 'w_train': w[train_idx],
        'smiles_train': smiles_arr[train_idx],
        'X_valid': X[valid_idx], 'y_valid': y[valid_idx], 'w_valid': w[valid_idx],
        'smiles_valid': smiles_arr[valid_idx],
        'X_test': X[test_idx], 'y_test': y[test_idx], 'w_test': w[test_idx],
        'smiles_test': smiles_arr[test_idx],
        'task_cols': task_cols,
        'invalid_smiles': invalid_smiles,
    }

Overwriting src/tools/data_prep.py


In [7]:
import importlib
import src.tools.data_prep
importlib.reload(src.tools.data_prep)
from src.tools.data_prep import load_tox21_clean

data = load_tox21_clean()
print(data['smiles_train'][:3])  # 처음 3개 SMILES 확인
print(data['smiles_train'].shape, data['X_train'].shape)  # 개수가 같은지 확인

[14:53:23] WARNING: not removing hydrogen atom without neighbors
[14:53:23] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:53:23] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:53:23] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:53:24] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:53:24] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:53:24] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:53:24] Explicit valence for atom # 16 Al, 6, is greater than permitted
[14:53:24] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[14:53:25] WARNING: not removing hydrogen atom without neighbors


['COCCC#N' 'N[C@@H](CSSC[C@H](N)C(=O)O)C(=O)O' 'O=C1CCCCC1']
(5476,) (5476, 2048)


In [8]:
X_train_emb = get_chemberta_embeddings(list(data['smiles_train']))
X_valid_emb = get_chemberta_embeddings(list(data['smiles_valid']))
X_test_emb = get_chemberta_embeddings(list(data['smiles_test']))

print("Train embedding shape:", X_train_emb.shape)
print("Valid embedding shape:", X_valid_emb.shape)

100%|██████████| 19/19 [00:04<00:00,  4.57it/s]

Train embedding shape: (5476, 768)
Valid embedding shape: (1173, 768)


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

y_train, w_train = data['y_train'], data['w_train']
y_valid, w_valid = data['y_valid'], data['w_valid']
task_cols = data['task_cols']

auc_scores_chemberta = {}

for i, task in enumerate(task_cols):
    train_mask = w_train[:, i] == 1
    valid_mask = w_valid[:, i] == 1

    clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    clf.fit(X_train_emb[train_mask], y_train[train_mask, i])

    probs = clf.predict_proba(X_valid_emb[valid_mask])[:, 1]
    auc = roc_auc_score(y_valid[valid_mask, i], probs)
    auc_scores_chemberta[task] = auc
    print(f"{task}: AUROC = {auc:.3f}")

print(f"\n평균 AUROC (ChemBERTa): {np.mean(list(auc_scores_chemberta.values())):.3f}")
print(f"평균 AUROC (ECFP baseline): 0.821  ← 비교 기준")

NR-AR: AUROC = 0.812
NR-AR-LBD: AUROC = 0.889
NR-AhR: AUROC = 0.849
NR-Aromatase: AUROC = 0.785
NR-ER: AUROC = 0.741
NR-ER-LBD: AUROC = 0.823
NR-PPAR-gamma: AUROC = 0.753
SR-ARE: AUROC = 0.734
SR-ATAD5: AUROC = 0.787
SR-HSE: AUROC = 0.695
SR-MMP: AUROC = 0.810
SR-p53: AUROC = 0.770

평균 AUROC (ChemBERTa): 0.787
평균 AUROC (ECFP baseline): 0.821  ← 비교 기준


In [11]:
import json
from datetime import datetime

comparison = {
    "ecfp_baseline": {
        "mean_auc": 0.8208144274160628,
        "auc_scores": 0.821,  # 이전 노트북에서 계산한 값이 이 세션에 없다면 0.821만 기록해도 됨
    },
    "chemberta_feature_extraction": {
        "mean_auc": float(np.mean(list(auc_scores_chemberta.values()))),
        "auc_scores": auc_scores_chemberta,
        "model": "seyonec/ChemBERTa-zinc-base-v1",
        "method": "CLS token embedding + RandomForest (frozen, no fine-tuning)",
    },
    "decision": "ECFP baseline 채택 (더 높은 AUROC, 더 빠른 학습/추론 속도)",
    "created_at": datetime.now().isoformat(),
}

with open('models/tox21_classifier/model_comparison.json', 'w') as f:
    json.dump(comparison, f, indent=2)

print("비교 결과 저장 완료")

비교 결과 저장 완료
